# Topic 5: Wide vs Narrow Transformations & Shuffle Internals

> **Phase 2 → Week 2 → Topic 5**

---

## The Sorting Hat Analogy (Harry Potter)

Imagine 1000 students arriving at Hogwarts:

**Narrow (no shuffle):**
- Each student gets a badge "Brave"/"Clever"/"Loyal"/"Cunning" based on their own traits
- No student needs to know about other students to get their badge
- Everyone gets processed in their own queue → **fast, parallel, no coordination**

**Wide (shuffle):**
- The Sorting Hat needs to balance house sizes: 250 per house
- It must look at ALL students first, count them, then decide assignments
- Students must move between queues based on the global decision → **coordination required**

Narrow = process each element independently → no coordination → no network transfer
Wide = result depends on multiple elements → coordination → data must move across the network = **shuffle**

---

## Narrow Transformations — The Fast Path

A transformation is **narrow** if each output partition depends on **at most one** input partition.

```
     Partition 0          Partition 1          Partition 2
  [1, 2, 3, 4, 5]     [6, 7, 8, 9, 10]    [11, 12, 13, 14]
        │                    │                     │
   filter(>3)          filter(>3)             filter(>3)
        │                    │                     │
   [4, 5]              [6, 7, 8, 9, 10]      [11, 12, 13, 14]

  Each output partition depends on EXACTLY ONE input partition.
  No data moves between machines. No network I/O.
```

**All narrow transformations:**
`map, flatMap, filter, mapValues, keys, values, union, sample, coalesce, mapPartitions, zip`

---

## Wide Transformations — The Shuffle Path

A transformation is **wide** if each output partition can depend on **multiple** input partitions.

```
     Partition 0           Partition 1          Partition 2
  [(A,1),(B,1),(C,1)]  [(A,1),(B,1),(A,1)]  [(C,1),(A,1)]
         │                    │                    │
         └──────────┬─────────┘                    │
                    │ groupByKey / reduceByKey       │
                    │ ALL 'A' values must go         │
                    │ to the SAME partition          │
                    │                               │
              New Partition 0   New Partition 1   New Partition 2
               (all 'A' rows)   (all 'B' rows)   (all 'C' rows)

  Each output partition receives data from ALL input partitions!
  Data moves across the network. This is the SHUFFLE.
```

**All wide transformations:**
`groupByKey, reduceByKey, aggregateByKey, sortByKey, sortBy, distinct, repartition, join, cogroup, intersection, subtract, cartesian`

---

## The Shuffle — What Actually Happens

A shuffle is the most expensive operation in Spark. Here's exactly what happens:

```
STEP 1: MAP SIDE (pre-shuffle)
  Each executor processes its partitions, applying narrow transformations.
  For a reduceByKey: pre-aggregates values with same key WITHIN each partition.
  Writes shuffle data to LOCAL DISK (shuffle write files, organized by target partition).
  This is "shuffle write" in the Spark UI.

STEP 2: NETWORK TRANSFER
  Executors pull the shuffle data they need from other executors' local disks.
  Data crosses the network here.
  This is the main cost: network bandwidth + serialization/deserialization.

STEP 3: REDUCE SIDE (post-shuffle)
  Each executor receives its shuffle data (all rows for its key range).
  Applies the actual aggregation/sort/group operation.
  Produces the final result partition.
  This is "shuffle read" in the Spark UI.
```

### Why Shuffles Are Expensive

1. **Disk I/O**: Shuffle data is written to executor local disk (to handle large datasets)
2. **Network I/O**: Data crosses the network between executors
3. **Serialization**: Data must be serialized to bytes for transfer
4. **Memory pressure**: If shuffle data doesn't fit in memory → disk spill
5. **Synchronization**: All map-side tasks must finish before reduce-side can start

### Stage Boundary

Every shuffle creates a **stage boundary**:
- Stage N ends when all map-side tasks complete and write shuffle data
- Stage N+1 starts when reduce-side tasks begin reading shuffle data
- Stages can't overlap — Stage N+1 can't start until ALL tasks in Stage N finish
  (even if just one task is slow → all reduce-side tasks wait)

---

## Key Metrics in Spark UI (Stages Tab)

| Metric | What it means | Bad sign |
|--------|--------------|----------|
| **Shuffle Write** | Bytes written by map-side tasks | Very high → too much data shuffled |
| **Shuffle Read** | Bytes read by reduce-side tasks | Should ≈ Shuffle Write |
| **Spill (memory)** | Data that couldn't fit in memory RAM | Non-zero → need more executor memory |
| **Spill (disk)** | Data spilled to executor local disk | Non-zero → serious memory pressure |
| **Task duration** | Time per task | Huge variation = data skew |
| **GC time** | Garbage collection | High % = JVM memory pressure |

---

## How Spark Decides Shuffle Partition Count

```python
# After a shuffle, how many partitions?
spark.conf.get("spark.sql.shuffle.partitions")  # default: 200

# For RDDs, it's:
sc.defaultParallelism  # or specify in reduceByKey(func, numPartitions=N)
```

**Tuning guidelines:**
- Too few partitions: huge tasks, slow, OOM
- Too many partitions: task scheduling overhead, many small files
- Rule of thumb: shuffle partitions ≈ 2-4× total executor cores
- For small datasets in local mode: set to 4-8 (default 200 is too many)

---

## Data Skew — The Silent Killer

Data skew happens when one key has FAR more values than others.

```
Key distribution (millions of records):
  'India'  → 50,000,000 records  ← SKEWED!
  'USA'    →  2,000,000 records
  'UK'     →    500,000 records
  'Others' →  1,000,000 records

With groupByKey:
  Partition 0 (India):  50M records → takes 10 minutes
  Partition 1 (USA):    2M records  → takes 30 seconds
  Partition 2 (UK):     500K records → takes 5 seconds
  Partition 3 (Others): 1M records  → takes 25 seconds

The ENTIRE STAGE waits for Partition 0 (India) to finish = 10 minutes.
3 cores idle for 9.5 minutes! Wasted resources.
```

**Fix: Salting (preview — covered in detail in Phase 4)**
```python
import random
# Add a random suffix to the key to distribute the load
rdd.map(lambda kv: (kv[0] + "_" + str(random.randint(0, 9)), kv[1]))
   .reduceByKey(lambda a, b: a + b)
   # Then remove the suffix and reduce again
   .map(lambda kv: (kv[0].rsplit("_", 1)[0], kv[1]))
   .reduceByKey(lambda a, b: a + b)
```

In [ ]:
from pyspark.sql import SparkSession
import time

spark = SparkSession.builder \
    .appName("Week2 - Wide vs Narrow") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("WARN")
print("SparkSession ready!")

## Part 1: Prove Narrow Transformations Stay in Same Stage

In [ ]:
rdd = sc.parallelize(range(1, 101), 4)

# Chain of narrow transformations
result = (
    rdd
    .filter(lambda x: x % 2 == 0)       # narrow
    .map(lambda x: x * 3)                # narrow
    .filter(lambda x: x > 50)           # narrow
    .map(lambda x: (x, x ** 2))          # narrow
)

# All these narrow transforms run in ONE stage
print("Narrow chain: filter → map → filter → map")
print("All happen in ONE stage (no shuffle)")
print(f"Result: {result.collect()[:5]}...")
print()

# Check lineage — no ShuffledRDD
print("Lineage (no ShuffledRDD = all narrow = one stage):")
print(result.toDebugString().decode())

## Part 2: See the Shuffle Boundary with `toDebugString()`

In [ ]:
# Each ShuffledRDD in toDebugString = one stage boundary

pairs = sc.parallelize([
    ("a", 1), ("b", 2), ("a", 3), ("c", 1),
    ("b", 4), ("c", 2), ("a", 5), ("b", 1),
], 4)

# ONE shuffle: reduceByKey
result1 = pairs.reduceByKey(lambda x, y: x + y)
print("After ONE shuffle (reduceByKey):")
print(result1.toDebugString().decode())
print("↑ One 'ShuffledRDD' = one stage boundary = 2 stages total")
print(f"Result: {sorted(result1.collect())}")

In [ ]:
# TWO shuffles: reduceByKey + sortByKey
result2 = pairs.reduceByKey(lambda x, y: x + y).sortByKey()
print("After TWO shuffles (reduceByKey + sortByKey):")
print(result2.toDebugString().decode())
print("↑ Two 'ShuffledRDD' = two stage boundaries = 3 stages total")
print(f"Result: {result2.collect()}")

## Part 3: Time Narrow vs Wide Transformations

In [ ]:
import time

n = 500_000
data = sc.parallelize(range(n), 4)
pairs = data.map(lambda x: (x % 100, x))  # 100 unique keys

print(f"Dataset: {n:,} elements, 100 unique keys, 4 partitions")
print()

# Time narrow: map + filter (no shuffle)
t0 = time.time()
narrow_result = data.map(lambda x: x*2).filter(lambda x: x > n).count()
narrow_time = time.time() - t0
print(f"Narrow (map + filter + count):     {narrow_time:.3f}s  [no shuffle, 1 stage]")

# Time wide: reduceByKey (shuffle)
t0 = time.time()
wide_result = pairs.reduceByKey(lambda a, b: a + b).count()
wide_time = time.time() - t0
print(f"Wide (reduceByKey + count):        {wide_time:.3f}s  [shuffle, 2 stages]")

# Time wider: reduceByKey + sortByKey (2 shuffles)
t0 = time.time()
wider_result = pairs.reduceByKey(lambda a, b: a + b).sortByKey().count()
wider_time = time.time() - t0
print(f"Wider (reduceByKey + sortByKey):   {wider_time:.3f}s  [2 shuffles, 3 stages]")

print()
print(f"Each shuffle adds overhead: shuffle 1 = {wide_time/narrow_time:.1f}x slower than narrow")

## Part 4: Data Skew Simulation

In [ ]:
# Simulate a skewed dataset where one key dominates

# Balanced dataset
balanced = sc.parallelize(
    [(f"key_{i%5}", 1) for i in range(50_000)], 4
)

# Skewed dataset: key_0 has 40k records, others have 2.5k each
skewed = sc.parallelize(
    [("key_0", 1)] * 40_000 +  # 80% of data!
    [(f"key_{i%4+1}", 1) for i in range(10_000)],
    4
)

t0 = time.time()
balanced.reduceByKey(lambda a, b: a + b).collect()
bal_time = time.time() - t0

t0 = time.time()
skewed.reduceByKey(lambda a, b: a + b).collect()
skew_time = time.time() - t0

print("Data Skew Impact:")
print(f"  Balanced (10k per key):    {bal_time:.3f}s")
print(f"  Skewed (40k for key_0):    {skew_time:.3f}s")
print()
print("Distribution in skewed dataset:")
counts = skewed.countByKey()
for k, v in sorted(counts.items()):
    bar = '█' * (v // 1000)
    print(f"  {k}: {v:,}  {bar}")
print()
print("In a cluster, the partition holding key_0 takes much longer.")
print("Other executors finish early and sit IDLE waiting for key_0.")
print("This is data skew. Detect it in Spark UI → Tasks → look for outlier task times.")

## Part 5: Salting — Fix Data Skew

In [ ]:
import random

# Skewed data — key_0 is dominant
skewed = sc.parallelize(
    [("key_0", 1)] * 40_000 +
    [(f"key_{i%4+1}", 1) for i in range(10_000)],
    4
)

print("=== Salting Fix for Data Skew ===")
print()
print("Without salting: key_0 → 40,000 records on ONE partition")
print("With salting:    key_0 split into key_0_0, key_0_1 ... key_0_9")
print("                 → 4,000 records each on 10 partitions!")
print()

# Step 1: Add salt (random suffix) to distribute
SALT_FACTOR = 4  # split each key into 4 sub-keys

# Add random salt to each key
salted = skewed.map(lambda kv: (f"{kv[0]}_{random.randint(0, SALT_FACTOR-1)}", kv[1]))

# Step 2: First reduce (with salt — distributed!)
first_reduce = salted.reduceByKey(lambda a, b: a + b)

# Step 3: Remove salt and reduce again
final = (
    first_reduce
    .map(lambda kv: (kv[0].rsplit("_", 1)[0], kv[1]))  # remove salt
    .reduceByKey(lambda a, b: a + b)                      # final aggregation
)

result = sorted(final.collect())
print("Final result after salting:")
for k, v in result:
    bar = '█' * (v // 5000)
    print(f"  {k}: {v:,}  {bar}")
print()
print("Same correct counts! But now the shuffle was distributed.")

## Part 6: `repartition()` vs `coalesce()` — Controlling Partition Count

In [ ]:
rdd = sc.parallelize(range(1, 101), 8)  # 8 partitions

print(f"Original: {rdd.getNumPartitions()} partitions")
print()

# coalesce: reduce partitions (narrow — no full shuffle)
# Just merges adjacent partitions — faster but may be unbalanced
coalesced_2 = rdd.coalesce(2)
print(f"coalesce(2):      {coalesced_2.getNumPartitions()} partitions")
print(f"  Partition sizes: {[len(p) for p in coalesced_2.glom().collect()]}")
print(f"  Narrow (no full shuffle) — fast but may be unbalanced")

# repartition: change partitions with FULL SHUFFLE — always balanced
repartitioned_4 = rdd.repartition(4)
print(f"\nrepartition(4):   {repartitioned_4.getNumPartitions()} partitions")
print(f"  Partition sizes: {[len(p) for p in repartitioned_4.glom().collect()]}")
print(f"  Wide (full shuffle) — slower but balanced")

# INCREASE partitions: MUST use repartition (coalesce can't increase)
repartitioned_16 = rdd.repartition(16)
print(f"\nrepartition(16):  {repartitioned_16.getNumPartitions()} partitions")
print(f"  (coalesce(16) on an 8-partition RDD would just return 8 partitions)")

print()
print("Rule: Use coalesce() to reduce before writing output (avoid shuffle).")
print("      Use repartition() to increase parallelism or balance data.")

## Part 7: Shuffle Partition Count Tuning

In [ ]:
# After a shuffle, how many partitions?
# Controlled by: spark.sql.shuffle.partitions for DataFrames
#                numPartitions argument for RDDs

data = sc.parallelize([(k % 5, v) for k, v in enumerate(range(10_000))], 4)

print("RDD shuffle partition count:")

# Default: uses sc.defaultParallelism
result_default = data.reduceByKey(lambda a, b: a + b)
print(f"  reduceByKey (default):          {result_default.getNumPartitions()} partitions")

# Explicit: pass numPartitions
result_8 = data.reduceByKey(lambda a, b: a + b, numPartitions=8)
print(f"  reduceByKey (numPartitions=8):  {result_8.getNumPartitions()} partitions")

result_2 = data.reduceByKey(lambda a, b: a + b, numPartitions=2)
print(f"  reduceByKey (numPartitions=2):  {result_2.getNumPartitions()} partitions")

print()
print("Guidelines:")
print("  Too few (1-2):  large tasks, OOM risk, underutilized cores")
print("  Too many (500+): task scheduling overhead, many tiny files")
print("  Sweet spot: 2-4x total executor cores")
print(f"  local[4] → use 4-16 shuffle partitions")
print(f"  10 executors × 4 cores = 40 cores → use 80-160 shuffle partitions")

## Part 8: Complete Pipeline — Narrow Then Wide, Minimizing Shuffles

In [ ]:
# Best practice: do ALL narrow transformations BEFORE the wide ones
# This reduces the amount of data that has to be shuffled

raw_sales = sc.parallelize([
    "2024-01,Electronics,Alice,299.99",
    "2024-01,Clothing,Bob,49.99",
    "2024-01,Electronics,Alice,599.99",
    "2024-02,Electronics,Bob,1299.99",
    "2024-02,Food,Carol,24.99",
    "2024-02,Clothing,Alice,79.99",
    "2024-02,Electronics,Carol,449.99",
    "2024-03,Food,Bob,34.99",
    "2024-03,Electronics,Alice,899.99",
    "2024-01,Food,Bob,15.99",
], 4)

# GOOD: filter and transform BEFORE shuffle
good_pipeline = (
    raw_sales
    .map(lambda l: l.split(","))                  # NARROW: parse (no shuffle)
    .filter(lambda f: f[1] == "Electronics")       # NARROW: filter early! (reduces shuffle data)
    .map(lambda f: (f[0], float(f[3])))            # NARROW: extract (month, amount)
    .reduceByKey(lambda a, b: a + b)               # WIDE: shuffle (but on SMALLER data)
    .sortByKey()                                   # WIDE: shuffle
)

print("Electronics revenue by month:")
for month, total in good_pipeline.collect():
    print(f"  {month}  ${total:,.2f}")

print()
print("Why this is optimal:")
print("  - Filter happens BEFORE shuffle (reduces rows from 10 to 5 before network transfer)")
print("  - map() extracts only needed fields before shuffle (fewer bytes transferred)")
print("  - Only 2 shuffles (reduceByKey + sortByKey) — minimum needed")
print("  - Catalyst Optimizer in DataFrames would reorder this automatically")
print("  - With RDDs: YOU must order it correctly")

## Complete Summary Table

| | Narrow | Wide |
|---|--------|------|
| Output partition depends on | 1 input partition | Multiple input partitions |
| Data movement | None | Full network shuffle |
| Stage boundary | No | Yes |
| Examples | map, filter, flatMap, union, coalesce | groupByKey, reduceByKey, join, distinct, repartition, sortBy |
| Speed | Fast | Slower (I/O bound) |
| Memory risk | Low | High (spill if data doesn't fit) |
| Skew risk | None | Yes (uneven key distribution) |

---

## Interview Cheat Sheet

**Q: What is the difference between narrow and wide transformations?**
> Narrow: each output partition depends on at most one input partition — no data movement, no shuffle, stays in the same stage. Wide: each output partition can depend on multiple input partitions — requires a shuffle (data redistribution across the network), creates a new stage boundary.

**Q: What happens during a shuffle?**
> Map-side tasks write shuffle files to local disk organized by target partition. Reduce-side tasks pull those files over the network from other executors. The data is then aggregated/sorted/grouped. Key costs: local disk I/O, network transfer, serialization/deserialization, and synchronization (all map tasks must finish before reduce tasks can start).

**Q: How do you detect and fix data skew?**
> Detect: In Spark UI → Stages → Tasks, look for one task taking much longer than others (e.g., task 0 takes 10 min, all others take 30 sec). Fix: Salting — add a random suffix to hot keys to distribute them, do a two-phase reduceByKey (first with salt, then without). Or in DataFrames: use AQE (Adaptive Query Execution) with skew join optimization (Phase 4 topic).

**Q: When should you use `repartition()` vs `coalesce()`?**
> `coalesce(n)` merges partitions without a full shuffle — use it to reduce partition count (e.g., before writing output). It's a narrow transformation, fast, but can be unbalanced. `repartition(n)` does a full shuffle — use when you need to increase partitions or need balanced distribution. The shuffle makes it slower but produces even partitions.

**Q: What is `spark.sql.shuffle.partitions` and what should you set it to?**
> It controls the number of partitions created after any shuffle operation for DataFrames/SQL. Default is 200, which is too high for small data (wasteful overhead) and may be too low for very large data. Rule of thumb: set to 2-4× total executor cores. For local testing, set to 4-16. For production (e.g., 50 executors × 4 cores = 200 cores), set to 400-800.

## Exercises

1. Create an RDD of 100 elements with 10 unique keys. Use `glom()` to see how `repartition(8)` vs `coalesce(2)` distributes the data.
2. Build a pipeline that processes a list of tuples. Count how many shuffles (ShuffledRDDs in `toDebugString()`) your pipeline has. Then optimize it to reduce shuffles.
3. Create a skewed dataset (80% one key, 20% others). Time `reduceByKey` with and without salting. Observe the difference.
4. Change `numPartitions` in `reduceByKey` from 2 to 4 to 8 on a 200-element dataset. How does it affect partition size distribution after the shuffle?

In [ ]:
# Exercise 1: repartition vs coalesce distribution
rdd = sc.parallelize(range(100), 10)

print("Exercise 1 — Partition distributions:")
print(f"\nOriginal (10 parts):")
print(f"  Sizes: {[len(p) for p in rdd.glom().collect()]}")

print(f"\nrepartition(4) [full shuffle]:")
rp = rdd.repartition(4)
print(f"  Sizes: {[len(p) for p in rp.glom().collect()]} ← balanced")

print(f"\ncoalesce(4) [no shuffle]:")
co = rdd.coalesce(4)
print(f"  Sizes: {[len(p) for p in co.glom().collect()]} ← may be unbalanced")

print(f"\ncoalesce(2) [no shuffle]:")
co2 = rdd.coalesce(2)
print(f"  Sizes: {[len(p) for p in co2.glom().collect()]}")

In [ ]:
# Exercise 2: Count shuffles in a pipeline
pairs = sc.parallelize([(i%5, i) for i in range(20)], 3)

# Pipeline with multiple shuffles
result = pairs.reduceByKey(lambda a,b: a+b).sortByKey()

print("Exercise 2 — Counting shuffles via toDebugString:")
debug_str = result.toDebugString().decode()
shuffles = debug_str.count("ShuffledRDD")
print(f"Shuffles detected: {shuffles}")
print("(Each 'ShuffledRDD' = 1 stage boundary = 1 shuffle operation)")